# FractureGuard — Milestone 2 GPU training (Kaggle)

Runs the frozen Milestone 2 protocol: YOLO26n preprocessing screen (3 variants × 25 epochs),
then YOLO11n / YOLO26n / YOLO26s on the selected variant (100 epochs, patience 20).
**Validation split only — the internal test split is not in the bundle and is never touched.**

**Inputs to attach (Add Input):**
1. The `fractureguard-m2-bundle` dataset (upload `work/kaggle/fractureguard_m2_bundle.zip`).
2. A GRAZPEDWRI-DX image dataset (16-bit PNGs, any of the `images/` or `images_part1..4` layouts).
   Every image is checked against the locally audited SHA-256 — a different copy is rejected.
3. *(Sessions 2+)* the output of the previous version of this notebook, so finished runs are
   skipped and interrupted runs resume from `last.pt`.

**Settings:** GPU T4, Internet on (for `pip install ultralytics` and weight downloads).

**Suggested sessions** (each stays under Kaggle's 12 h limit; use *Save Version → Save & Run All*):

| Session | `STAGE` | `MODELS` |
|---|---|---|
| 1 | `"preprocess"` | — |
| 2 | `"detectors"` | `["yolo11n", "yolo26n"]` |
| 3 | `"detectors"` | `["yolo26s"]` |

Re-running a session is always safe: completed experiments are skipped.

In [ ]:
# ---- Session settings -------------------------------------------------------------
STAGE = "preprocess"   # "preprocess", "detectors", or "all"
MODELS = None          # e.g. ["yolo11n", "yolo26n"]; None = all three (detectors stage)
MAX_HOURS = 10.5       # do not START a new experiment after this many hours
DEVICE = "0"           # "0" = one T4; "0,1" = both T4s via DDP (faster, same total batch)
WORKERS = 4

In [ ]:
import hashlib, json, os, shutil, subprocess, sys, zipfile
from pathlib import Path

INPUT, WORK = Path("/kaggle/input"), Path("/kaggle/working")
PROJECT = WORK / "fractureguard"
DERIVED = Path("/tmp/fg_derived")   # rebuilt each session; kept out of the saved output

def find_first(name):
    return next(iter(INPUT.rglob(name)), None)

manifest_path = find_first("BUNDLE_MANIFEST.json")
if manifest_path is not None:
    bundle_root = manifest_path.parent
else:
    archive = find_first("fractureguard_m2_bundle.zip")
    if archive is None:
        raise FileNotFoundError("Attach the fractureguard-m2-bundle dataset")
    bundle_root = Path("/tmp/fg_bundle")
    shutil.rmtree(bundle_root, ignore_errors=True)
    zipfile.ZipFile(archive).extractall(bundle_root)
    manifest_path = bundle_root / "BUNDLE_MANIFEST.json"

manifest = json.loads(manifest_path.read_text())
bad = []
for rel, digest in manifest["files"].items():
    source = bundle_root / rel
    target = PROJECT / rel
    target.parent.mkdir(parents=True, exist_ok=True)
    shutil.copyfile(source, target)  # code only; existing reports/ are left alone
    if hashlib.sha256(target.read_bytes()).hexdigest() != digest:
        bad.append(rel)
if bad:
    raise RuntimeError(f"Bundle files failed SHA-256 check: {bad[:5]}")
print(f"Bundle OK: {len(manifest['files'])} files, created {manifest['created_utc']}")

In [ ]:
env = dict(os.environ)
env["PYTHONPATH"] = str(PROJECT / "src")
env["YOLO_CONFIG_DIR"] = str(WORK / "ultralytics_config")
try:
    import ultralytics  # noqa: F401
except ImportError:
    if subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics>=8.4,<9"]).returncode:
        offline = find_first("ultralytics/__init__.py")
        if offline is None:
            raise RuntimeError("No internet and no offline Ultralytics bundle attached")
        env["PYTHONPATH"] += os.pathsep + str(offline.parent.parent)
for weights in ("yolo11n.pt", "yolo26n.pt", "yolo26s.pt"):   # optional offline weights
    found = find_first(weights)
    if found is not None and not (PROJECT / weights).exists():
        shutil.copyfile(found, PROJECT / weights)
subprocess.run(
    [sys.executable, "-c",
     "import torch, ultralytics; print('ultralytics', ultralytics.__version__, '| torch', "
     "torch.__version__, '| GPUs', [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])"],
    env=env, check=True,
)

In [ ]:
# Locate the GRAZPEDWRI-DX source from the first cohort image and preflight every path.
sys.path.insert(0, str(PROJECT / "src"))
import pandas as pd
from fractureguard.data.training_data import _resolve_source_image

cohort = pd.read_csv(PROJECT / "data/manifests/grazpedwri_m2_cohort.csv")
probe = find_first(Path(cohort["image_path"].iloc[0]).name)
if probe is None:
    raise FileNotFoundError("Attach a GRAZPEDWRI-DX image dataset")
DATASET_ROOT = probe.parent
while DATASET_ROOT.name.lower().startswith("images"):
    DATASET_ROOT = DATASET_ROOT.parent
missing = []
for image_path in cohort["image_path"]:
    try:
        _resolve_source_image(DATASET_ROOT, image_path)
    except FileNotFoundError:
        missing.append(image_path)
print(f"Source root: {DATASET_ROOT} | cohort images: {len(cohort)} | missing: {len(missing)}")
if missing:
    raise FileNotFoundError(f"{len(missing)} cohort images are missing, e.g. {missing[:5]}")

In [ ]:
# Restore finished/interrupted runs from a previous notebook version, if attached.
for state in INPUT.rglob("reports/m2_training_selection.json"):
    previous_reports = state.parent
    shutil.copytree(previous_reports, PROJECT / "reports", dirs_exist_ok=True)
    print(f"Restored previous reports from {previous_reports}")
    break
else:
    print("No previous state attached (expected for session 1)")

In [ ]:
command = [
    sys.executable, "scripts/run_m2_pipeline.py",
    "--dataset-root", str(DATASET_ROOT),
    "--variants-root", str(DERIVED),
    "--stage", STAGE,
    "--device", DEVICE,
    "--workers", str(WORKERS),
    "--max-hours", str(MAX_HOURS),
]
if MODELS:
    command += ["--models", *MODELS]
print(" ".join(command))
process = subprocess.Popen(command, cwd=PROJECT, env=env, stdout=subprocess.PIPE,
                           stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in process.stdout:
    print(line, end="")
if process.wait():
    raise RuntimeError(f"Pipeline exited with code {process.returncode}")

In [ ]:
# Summary + a compact download (cards, results, best/last weights; no per-epoch checkpoints).
state = json.loads((PROJECT / "reports/m2_training_selection.json").read_text())
rows = [
    {"stage": stage, "name": name, "action": rec.get("action"),
     "epochs": rec.get("epochs_completed"), "best_epoch": rec.get("best_epoch"),
     "best_mAP50-95": rec.get("best_mAP50-95")}
    for stage in ("preprocessing", "detectors") for name, rec in state.get(stage, {}).items()
]
display(pd.DataFrame(rows))
print("selected_preprocessing:", state.get("selected_preprocessing"))
print("best_validation_detector:", state.get("best_validation_detector"))

archive = WORK / "fractureguard_m2_reports.zip"
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as handle:
    for path in (PROJECT / "reports").rglob("*"):
        if path.is_file() and not (path.suffix == ".pt" and path.stem.startswith("epoch")):
            handle.write(path, path.relative_to(PROJECT).as_posix())
print(f"Download: {archive} ({archive.stat().st_size / 1e6:.1f} MB)")